# ============================================
# MEMBER 2: BASELINE MODELS
# Bank Telemarketing Prediction
# With SMOTE vs Without SMOTE | All 4 Scenarios
# ============================================

In [20]:
# ============================================
# MEMBER 2: BASELINE MODELS
# Bank Telemarketing Prediction
# USING ORIGINAL FEATURES ONLY (NO ENGINEERED FEATURES)
# ============================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("MEMBER 2: BASELINE MODELS (ORIGINAL FEATURES ONLY)")
print("=" * 60)

# ============================================
# 1. LOAD ORIGINAL DATA
# ============================================

# Load original cleaned data (before Member 1's feature engineering)
df = pd.read_csv('data/processed/bankclean.csv')

# Original features only
original_features = ['age', 'job', 'marital', 'education', 'default', 
                     'balance', 'housing', 'loan', 'contact', 'month', 
                     'duration', 'campaign', 'pdays', 'previous', 'poutcome']

# Keep only original features
df_original = df[original_features + ['y']].copy()

# Convert target to binary
y = (df_original['y'] == 'yes').astype(int)
X = df_original.drop('y', axis=1)

print("✅ Data loaded (original features only)")
print(f"Total samples: {X.shape}")
print(f"Features: {X.columns.tolist()}")
print(f"Target: yes={sum(y)}, no={len(y)-sum(y)}")

# ============================================
# 2. CREATE TRAIN/TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target: yes={sum(y_train)}, no={len(y_train)-sum(y_train)}")
print(f"Test target: yes={sum(y_test)}, no={len(y_test)-sum(y_test)}")

# ============================================
# 3. CREATE 4 SCENARIOS
# ============================================

all_features = X_train.columns.tolist()

# S1: All features (with duration)
X_train_S1 = X_train.copy()
X_test_S1 = X_test.copy()

# S2: No duration (realistic)
duration_cols = ['duration']
X_train_S2 = X_train.drop(columns=duration_cols)
X_test_S2 = X_test.drop(columns=duration_cols)

# S3: Demographics only
demo_cols = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan']
X_train_S3 = X_train[demo_cols]
X_test_S3 = X_test[demo_cols]

# S4: Previous campaign only
prev_cols = ['pdays', 'previous', 'poutcome']
X_train_S4 = X_train[prev_cols]
X_test_S4 = X_test[prev_cols]

print("\n" + "=" * 60)
print("4 SCENARIOS READY")
print("=" * 60)
print(f"S1 (All features with duration): {X_train_S1.shape}")
print(f"S2 (No duration - realistic): {X_train_S2.shape}")
print(f"S3 (Demographics only): {X_train_S3.shape}")
print(f"S4 (Previous campaign only): {X_train_S4.shape}")

# ============================================
# 4. PREPROCESSING FUNCTION
# ============================================

def create_preprocessor(X):
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    return ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# ============================================
# 5. EVALUATION FUNCTION
# ============================================

def evaluate_model(X_train, X_test, y_train, y_test, model, model_name, scenario_name, use_smote=False):
    preprocessor = create_preprocessor(X_train)
    
    if use_smote:
        pipeline = ImbPipeline([
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=42)),
            ('classifier', model)
        ])
    else:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    return {
        'Model': model_name,
        'Scenario': scenario_name,
        'SMOTE': 'Yes' if use_smote else 'No',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }

# ============================================
# 6. DEFINE MODELS
# ============================================

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Lasso (L1)': LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42),
    'Ridge (L2)': LogisticRegression(penalty='l2', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=50)
}

# ============================================
# 7. SCENARIOS DICTIONARY
# ============================================

scenarios = {
    'S1_All_WithDuration': (X_train_S1, X_test_S1),
    'S2_Realistic_NoDuration': (X_train_S2, X_test_S2),
    'S3_DemographicsOnly': (X_train_S3, X_test_S3),
    'S4_PreviousOnly': (X_train_S4, X_test_S4)
}

# ============================================
# 8. RUN ALL MODELS
# ============================================

print("\n" + "=" * 60)
print("RUNNING MODELS (4 models × 4 scenarios × 2 SMOTE = 32 runs)")
print("=" * 60)

results = []

for scenario_name, (X_train_sc, X_test_sc) in scenarios.items():
    print(f"\n{'='*50}")
    print(f"SCENARIO: {scenario_name}")
    print(f"{'='*50}")
    
    for model_name, model in models.items():
        print(f"  Running {model_name}...")
        
        # Without SMOTE
        result_no = evaluate_model(
            X_train_sc, X_test_sc, y_train, y_test,
            model, model_name, scenario_name, use_smote=False
        )
        results.append(result_no)
        print(f"    No SMOTE - Recall: {result_no['Recall']:.4f}, AUC: {result_no['ROC-AUC']:.4f}")
        
        # With SMOTE
        result_smote = evaluate_model(
            X_train_sc, X_test_sc, y_train, y_test,
            model, model_name, scenario_name, use_smote=True
        )
        results.append(result_smote)
        print(f"    With SMOTE - Recall: {result_smote['Recall']:.4f}, AUC: {result_smote['ROC-AUC']:.4f}")

# ============================================
# 9. CONVERT TO DATAFRAME
# ============================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results_df.insert(0, 'Rank', results_df.index + 1)

print("\n" + "=" * 60)
print("RESULTS COMPLETE!")
print("=" * 60)
print(f"Total model runs: {len(results_df)}")

# ============================================
# 10. SAVE RESULTS
# ============================================

os.makedirs('results/tables', exist_ok=True)
results_df.to_csv('results/tables/baseline_results_original_features.csv', index=False)
print(f"\n✓ Saved: results/tables/baseline_results_original_features.csv")

# ============================================
# 11. DISPLAY TOP RESULTS
# ============================================

print("\n" + "=" * 60)
print("TOP 15 RESULTS (Sorted by ROC-AUC)")
print("=" * 60)
print(results_df.head(15).to_string())

# ============================================
# 12. SMOTE IMPACT (S2 - Realistic Scenario)
# ============================================

s2_results = results_df[results_df['Scenario'] == 'S2_Realistic_NoDuration']
if len(s2_results) > 0:
    print("\n" + "=" * 60)
    print("SMOTE IMPACT ON RECALL (S2 - Realistic Scenario)")
    print("=" * 60)
    
    smote_impact = s2_results.pivot_table(
        index='Model', 
        columns='SMOTE', 
        values='Recall'
    ).round(4)
    
    if 'Yes' in smote_impact.columns and 'No' in smote_impact.columns:
        smote_impact['Improvement'] = smote_impact['Yes'] - smote_impact['No']
        smote_impact['Improvement_%'] = (smote_impact['Improvement'] / smote_impact['No'] * 100).round(1)
        print(smote_impact.to_string())

# ============================================
# 13. BEST MODEL
# ============================================

print("\n" + "=" * 60)
print("BEST MODEL OVERALL")
print("=" * 60)
best = results_df.iloc[0]
print(f"  Model: {best['Model']}")
print(f"  Scenario: {best['Scenario']}")
print(f"  SMOTE: {best['SMOTE']}")
print(f"  ROC-AUC: {best['ROC-AUC']:.4f}")
print(f"  Recall: {best['Recall']:.4f}")
print(f"  Precision: {best['Precision']:.4f}")
print(f"  F1: {best['F1']:.4f}")

print("\n" + "=" * 60)
print("MEMBER 2 - BASELINE MODELS COMPLETE!")
print("=" * 60)

MEMBER 2: BASELINE MODELS (ORIGINAL FEATURES ONLY)
✅ Data loaded (original features only)
Total samples: (45211, 15)
Features: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']
Target: yes=5289, no=39922

Train: (36168, 15), Test: (9043, 15)
Train target: yes=4231, no=31937
Test target: yes=1058, no=7985

4 SCENARIOS READY
S1 (All features with duration): (36168, 15)
S2 (No duration - realistic): (36168, 14)
S3 (Demographics only): (36168, 8)
S4 (Previous campaign only): (36168, 3)

RUNNING MODELS (4 models × 4 scenarios × 2 SMOTE = 32 runs)

SCENARIO: S1_All_WithDuration
  Running Logistic Regression...
    No SMOTE - Recall: 0.3478, AUC: 0.9058
    With SMOTE - Recall: 0.8072, AUC: 0.9066
  Running Lasso (L1)...
    No SMOTE - Recall: 0.3488, AUC: 0.9058
    With SMOTE - Recall: 0.8072, AUC: 0.9066
  Running Ridge (L2)...
    No SMOTE - Recall: 0.3478, AUC: 0.9058
    With SMOTE